# Major Project: Seasonal Agriculture Performance Analysis

**Name:** Ankana Paul

**College Name:** Techno International Batanagar

**AICTE STU ID:** STU674aa6f6f33431732945654

**Project:** Seasonal Agriculture Performance Analysis  

**Program:** VOIS for Tech Program on DATA ANALYTICS

## Objective

The objective of this project is to analyze agricultural performance across different seasons and identify meaningful patterns, trends, relationships and variations in the available dataset.

This analysis focuses on seasonal differences in agricultural production, environmental conditions, resource usage and economic performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway, pearsonr

df = pd.read_csv("seasonal_agriculture_performance_dataset.csv")
df.shape, df.head()


## 1. Dataset overview

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.info())
display(df.describe(include="all").T)


### Key analytical questions
1. How does yield vary across Kharif, Rabi and Zaid?
2. How do revenue, cost and profit differ by season?
3. How do rainfall, soil moisture and disease/pest risk change across seasons?
4. Does water use/efficiency differ by season?
5. Which variables show the strongest association with yield?
6. Do irrigation methods show different yield patterns across seasons?


## 2. Data quality and cleaning

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)
display(missing[missing > 0])

# Median imputation for numeric fields with missing values.
clean = df.copy()
for col in ["Rainfall_mm", "Soil_Moisture_pct", "Yield_Tonnes_Ha"]:
    clean[col] = clean[col].fillna(clean[col].median())

print("\nMissing values after cleaning:")
display(clean.isna().sum()[clean.isna().sum() > 0])


**Cleaning decision:** The dataset contains missing values in rainfall, soil moisture and yield, with no duplicate rows. Because these are numeric variables and the project is comparative/analytical, median imputation is used to reduce sensitivity to extreme values while retaining all observations. 

## 3. Seasonal distribution

In [ ]:
season_order = ["Kharif", "Rabi", "Zaid"]
display(clean["Season"].value_counts().reindex(season_order))

clean["Season"].value_counts().reindex(season_order).plot(kind="bar", title="Number of Farms by Season")
plt.ylabel("Number of farms")
plt.tight_layout()
plt.show()


## 4. Seasonal performance

In [ ]:
metrics = [
    "Yield_Tonnes_Ha", "Production_Tonnes", "Revenue_INR",
    "Total_Cost_INR", "Profit_INR", "Water_Efficiency_t_per_1000m3",
    "Water_Used_m3", "Disease_Pest_Risk_pct"
]
season_summary = clean.groupby("Season")[metrics].mean().reindex(season_order)
display(season_summary.round(2))


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
season_summary["Yield_Tonnes_Ha"].plot(kind="bar", ax=ax, title="Average Yield by Season")
ax.set_ylabel("Yield (tonnes/ha)")
ax.set_xlabel("Season")
plt.tight_layout()
plt.show()


**Interpretation:** In this dataset, average yield is highest in Kharif (5.63 t/ha), followed by Rabi (5.04) and Zaid (4.64). However, the one-way ANOVA p-value for yield is approximately 0.214, so the observed mean differences are not statistically significant at the 5% level. This indicates that although Kharif has a higher average yield, the difference between seasons is not statistically significant.


## 5. Economic outcomes

In [ ]:
economic = clean.groupby("Season")[["Revenue_INR","Total_Cost_INR","Profit_INR"]].mean().reindex(season_order)
display(economic.round(2))

positive_profit = (clean["Profit_INR"] > 0).groupby(clean["Season"]).mean().reindex(season_order)
display((positive_profit * 100).round(1).rename("Positive profit rate (%)"))


In [ ]:
economic.plot(kind="bar", figsize=(10,5), title="Average Revenue, Cost and Profit by Season")
plt.ylabel("INR per farm")
plt.xlabel("Season")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Interpretation:** Kharif has the strongest average economic outcome: mean profit is about ₹178,915 per farm and about 57.8% of Kharif observations have positive profit. Rabi averages about ₹87,689 profit with a 48.9% positive-profit rate, while Zaid averages a loss of about ₹24,805 with a 35.5% positive-profit rate. ANOVA indicates statistically significant differences in profit across seasons (p < 0.001).


## 6. Environmental conditions and risk

In [ ]:
environment = clean.groupby("Season")[[
    "Rainfall_mm", "Avg_Temperature_C", "Humidity_pct",
    "Soil_Moisture_pct", "Disease_Pest_Risk_pct"
]].mean().reindex(season_order)
display(environment.round(2))

environment[["Rainfall_mm","Soil_Moisture_pct","Disease_Pest_Risk_pct"]].plot(
    marker="o", figsize=(10,5), title="Seasonal Environmental Conditions and Risk"
)
plt.ylabel("Average value")
plt.tight_layout()
plt.show()


**Interpretation:** Kharif has the highest average rainfall (849.20 mm) and soil moisture (31.15%), while Zaid has the lowest (304.65 mm and 19.28%). Disease/pest risk is also highest in Kharif (54.47%) and lowest in Zaid (38.22%). These results show seasonal differences in the dataset, but they do not establish a direct cause-and-effect relationship.


## 7. Water use and irrigation

In [ ]:
water = clean.groupby("Season")[[
    "Water_Used_m3", "Water_Efficiency_t_per_1000m3"
]].mean().reindex(season_order)
display(water.round(2))

irrigation_yield = clean.groupby(["Season","Irrigation_Method"])["Yield_Tonnes_Ha"].mean().unstack()
display(irrigation_yield.reindex(season_order).round(2))


In [ ]:
water.plot(kind="bar", figsize=(10,5), title="Seasonal Water Use and Efficiency")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

irrigation_yield.reindex(season_order).plot(kind="bar", figsize=(10,5), title="Average Yield by Season and Irrigation Method")
plt.ylabel("Yield (tonnes/ha)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Interpretation:** Average water efficiency is highest in Kharif (5.89 t/1000m³), followed by Rabi (5.19) and Zaid (4.41). Across the supplied data, Drip irrigation has the highest average yield in each season, although this comparison is observational and does not control for crop, farm size, region or other confounders.


## 8. Relationship analysis

In [ ]:
corr_vars = [
    "Rainfall_mm", "Avg_Temperature_C", "Humidity_pct", "Sunlight_Hours_Day",
    "Soil_pH", "Soil_Moisture_pct", "Nitrogen_kg_ha", "Phosphorus_kg_ha",
    "Potassium_kg_ha", "Fertilizer_kg_ha", "Pesticide_Litre_ha",
    "Seed_Quality_Score", "Water_Used_m3",
    "Water_Efficiency_t_per_1000m3", "Disease_Pest_Risk_pct",
    "Market_Price_INR_Tonne", "Total_Cost_INR"
]

rows = []
for col in corr_vars:
    r, p = pearsonr(clean[col], clean["Yield_Tonnes_Ha"])
    rows.append([col, r, p])

corr = pd.DataFrame(rows, columns=["Variable","Pearson_r","p_value"])
corr["abs_r"] = corr["Pearson_r"].abs()
display(corr.sort_values("abs_r", ascending=False).drop(columns="abs_r").head(10))


In [ ]:
top = corr.sort_values("abs_r", ascending=False).head(8).sort_values("Pearson_r")
plt.figure(figsize=(10,5.5))
plt.barh(top["Variable"], top["Pearson_r"])
plt.axvline(0, linewidth=0.8)
plt.xlabel("Pearson correlation with yield")
plt.title("Variables Most Strongly Associated with Yield")
plt.tight_layout()
plt.show()


**Interpretation:** Water efficiency has the strongest positive linear association with yield in this dataset (r ≈ 0.913), while water used is moderately positive (r ≈ 0.386). Market price has a moderate negative association with yield (r ≈ -0.383). These correlations indicate association, not causation. The very strong water-efficiency/yield relationship should be interpreted carefully because water efficiency is itself an outcome/derived performance measure.


## 9. Statistical significance of selected seasonal differences

In [ ]:
for col in ["Yield_Tonnes_Ha","Profit_INR","Water_Efficiency_t_per_1000m3","Disease_Pest_Risk_pct"]:
    groups = [g[col].values for _, g in clean.groupby("Season")]
    stat, p = f_oneway(*groups)
    print(f"{col}: F={stat:.3f}, p={p:.6g}")


### Statistical conclusion
- **Yield:** p ≈ 0.214 → no statistically significant seasonal mean difference at α = 0.05.
- **Profit:** p < 0.001 → statistically significant seasonal differences.
- **Water efficiency:** p ≈ 0.00097 → statistically significant seasonal differences.
- **Disease/pest risk:** p < 0.001 → statistically significant seasonal differences.

Statistical significance does not establish causality; it only indicates that the observed group means differ more than expected under the ANOVA null hypothesis.


## 10. Conclusions and recommendations

### Conclusions
1. Kharif shows the highest average yield, revenue, profit and water efficiency in the supplied dataset.
2. Zaid has the weakest average economic performance and the lowest positive-profit rate.
3. Rainfall and soil moisture are substantially higher in Kharif than in Rabi and Zaid.
4. Disease/pest risk is highest in Kharif in the dataset.
5. Water efficiency is strongly associated with yield, but it should be treated as a performance measure rather than a proven causal driver.
6. Irrigation-method comparisons suggest higher observed yields for Drip irrigation, but confounding factors must be considered.

### Recommendations
- Prioritize seasonal planning using profitability and water-efficiency indicators, not yield alone.
- Investigate why Zaid has lower profitability and whether irrigation/resource costs are contributing.
- Evaluate efficient irrigation methods, especially Drip, while controlling for crop and regional differences.
- Use rainfall, soil moisture and pest-risk patterns to support seasonal resource planning.
- For future work, build predictive models using additional years, verified regional data and controlled experiments.


## 11. Limitations
- The analysis is based only on the supplied dataset.
- Missing numeric values were median-imputed.
- Correlation and ANOVA results do not establish causation.
- Crop, region, irrigation and other factors may confound seasonal comparisons.
- The dataset appears structured for analytical practice; real-world deployment would require validated field data.

## 12.Future Scope
- Integrate multi-year real-world agricultural and weather datasets.
- Build predictive models for yield, profitability and disease/pest risk.
- Develop an interactive dashboard for seasonal monitoring.
- Add crop- and region-specific recommendations.
- Use controlled/causal analysis to evaluate irrigation and input decisions.
